In [ ]:
import rasterio
from pathlib import Path
from tile_mate import get_raster_from_tiles
from dist_s1_enumerator import get_mgrs_table
from dem_stitcher.rio_tools import reproject_arr_to_match_profile
from rasterio.crs import CRS
from colormap_utils import create_random_colormap
from tqdm import tqdm
from rasterio.warp import transform_bounds
from shapely import box

In [ ]:
transform_bounds()

In [6]:
df_mgrs_all = get_mgrs_table()
df_mgrs_all.head()

,mgrs_tile_id,utm_epsg,utm_wkt,geometry
0,01FBE,32701,"MULTIPOLYGON(((199980 4500040,199980 4390240,3...","MULTIPOLYGON (((-179.63379 -49.62222, -179.688..."
1,01FBF,32701,"MULTIPOLYGON(((199980 4600000,199980 4490200,3...","MULTIPOLYGON (((-179.58653 -48.72398, -179.638..."
2,01GBH,32701,"MULTIPOLYGON(((199980 4800040,199980 4690240,3...","MULTIPOLYGON (((-179.49865 -46.9259, -179.5458..."
3,01GDM,32701,"MULTIPOLYGON(((399960 5200000,399960 5090200,5...","POLYGON ((-178.23431 -43.34619, -178.25485 -44..."
4,01GEL,32701,"MULTIPOLYGON(((499980 5100040,499980 4990240,6...","POLYGON ((-177.00025 -44.25288, -177.00025 -45..."


In [16]:
comp_dir = Path('aggregated_validation_dist_2024')
agg_dirs = sorted(list(comp_dir.glob('*/')))
mgrs_tile_ids = [d.stem.split('__')[-1] for d in agg_dirs]
mgrs_tile_ids[:3]

['14RNU', '16SBF', '22KFA']

In [ ]:
def get_lc_mask(ref_path):
    with rasterio.open(ref_path) as src:
        p_utm = src.profile
        bounds_utm = list(ds.bounds)
    bounds_utm = box(*bounds_utm).buffer(90).bounds
    bounds_4326 = transform_bounds(p_utm['crs'], CRS.from_epsg(4326), *bounds_utm)
    X_lc, p_lc = get_raster_from_tiles(bounds_4326, 'glad_landcover', year=2020)
    X_lc_utm, p_lc_utm = reproject_arr_to_match_profile(X_lc, p_lc, p_utm, resampling='nearest')
    return X_lc_utm, p_lc_utm

def serialize_lc_mask(dst_dir):
    ref_path = list(dst_dir.glob('dist_s1*.tif'))[0]
    X_lc_utm, p_lc_utm = get_lc_mask(ref_path)
    mgrs_tile_id = dst_dir.stem.split('__')[-1]
    dst_path = dst_dir / f'{mgrs_tile_id}_glad_lc_mask.tif'
    cmap = create_random_colormap(n_colors=256)
    with rasterio.open(dst_path, 'w', **p_lc_utm) as dst:
        dst.write(X_lc_utm)
        dst.write_colormap(1, cmap)
    return dst_path


In [4]:
paths = [serialize_lc_mask(mgrs_tile_id, agg_dir) for (mgrs_tile_id, agg_dir) in zip(mgrs_tile_ids, tqdm(agg_dirs))]

NameError: name 'mgrs_tile_ids' is not defined